# Modellierung & Evaluation

Verglichen werden klassische Pipelines und generative DeepTabular-Modelle, um die Hypothesen zu Kredit-Scoring-Daten zu testen:
- **H1:** Klassische ML-Modelle erzielen vergleichbare oder bessere Ergebnisse als generative Modelle auf Tabulardaten.
- **H2:** Generative Modelle verursachen einen deutlich höheren Ressourcenbedarf gemessen an Trainingszeit/Rechenaufwand.

## 1️⃣ Vorgehensweise

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from deeptabular.models import MambularClassifier, FTTransformerClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)
from scipy.stats import randint, uniform
import time

In [2]:
train_df = pd.read_csv("../data/train/train_fe.csv")
test_df = pd.read_csv("../data/test/test_fe.csv")

print(f"Trainigsdatensatz: {train_df.shape}")
print(f"Testdatensatz: {test_df.shape}")

Trainigsdatensatz: (72878, 60)
Testdatensatz: (50000, 59)


## 2️⃣ Train-Test Split

In [3]:
X = train_df.drop("Credit_Score", axis=1)
y = train_df["Credit_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

print(f"X_Trainigsdatensatz: {X_train.shape}")
print(f"X_Testdatensatz: {X_test.shape}")
print(f"y_Trainigsdatensatz: {y_train.shape}")
print(f"y_Testdatensatz: {X_test.shape}")

X_Trainigsdatensatz: (48828, 59)
X_Testdatensatz: (24050, 59)
y_Trainigsdatensatz: (48828,)
y_Testdatensatz: (24050, 59)


## 3️⃣ Pipelines

### Preprocessing: Numerische & Kategorische Pipelines

- Numerische Features enthalten alle kontinuierlichen und zählenden Engineering-Kennzahlen (z. B. Einkommens-, Delay- und Ratio-Variablen).
- Kategorisch bleiben `Month`, `Occupation` und `Credit_Mix`.
- Verarbeitung: Numerische Werte werden median-imputet und skaliert, kategorische Werte werden mit dem häufigsten Wert imputet und anschließend One-Hot-encodiert.


In [4]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

print(f"{len(num_cols)} numerische Merkmale und {len(cat_cols)} kategorielle Merkmale identifiziert.")
print(f"Kategorisch: {cat_cols}")
print(f"Numerisch: {num_cols}")

56 numerische Merkmale und 3 kategorielle Merkmale identifiziert.
Kategorisch: ['Month', 'Occupation', 'Credit_Mix']
Numerisch: ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 'Loan_Types_Count', 'Has_Not_Specified_Loan', 'LoanType_Credit-Builder_Loan', 'LoanType_Payday_Loan', 'LoanType_Student_Loan', 'LoanType_Home_Equity_Loan', 'LoanType_Debt_Consolidation_Loan', 'Payment_Spent_Level', 'Payment_Value_Level', 'Is_Payment_Behaviour_Unknown', 'Pays_Min_Amount_Score', 'Is_Min_Payment_Unknown', 'Credit_Mix_Score', 'Credit_History_Months', 'Credit_Age_Gap', 'Credit_History_to_Age', 'Delay_Impact', 'Delay_per_Loan', 'Delayed_Payment_Ratio', 'Debt_To_Income_Ratio', 'EMI_To_Income_Ratio',

In [5]:
numeric_pre = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

In [6]:
categorical_pre = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pre, num_cols),
        ("cat", categorical_pre, cat_cols),
    ],
    remainder="drop",
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### Modell-Pipelines

- **LR**: Logistic Regression als lineares Baseline-Modell.
- **RF**: RandomForestClassifier als robuster, baumbasierter Klassifikator.
- **HGB**: HistGradientBoostingClassifier als leistungsfähiger Gradient-Boosting-Ansatz.
- **MAM**: MambularClassifier aus DeepTabular für sequentielle Mamba-Blöcke.
- **FTT**: FTTransformerClassifier als attention-basiertes DeepTabular-Modell.


In [8]:
pipe_lr = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(
        random_state=42,
    )),
])

pipe_lr

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
pipe_rf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
    )),
])

pipe_rf

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
pipe_hgb = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", HistGradientBoostingClassifier(random_state=42)),
])

pipe_hgb

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Deeptabular-Modelle übernehmen eigenständig die Proprocessing-Schritte

In [11]:
pipe_mam = Pipeline(steps=[
    ("model", MambularClassifier(
        numerical_preprocessing="standardization",
        categorical_preprocessing="one-hot",
    )),
])

In [12]:
pipe_ftt = Pipeline(steps=[
    ("model", FTTransformerClassifier(
        numerical_preprocessing="standardization",
        categorical_preprocessing="one-hot",
    )),
])

## 4️⃣ Hyperparameter-Tuning

In [13]:
param_grid_lr = {
    "model__C": [0.1, 1.0],
    "model__class_weight": [None, "balanced"],
    "model__max_iter": [200, 500],
}

param_grid_rf = {
    "model__n_estimators": [200],
    "model__max_depth": [None, 20],
    "model__min_samples_leaf": [1, 4],
    "model__class_weight": [None, "balanced"],
}

param_grid_hgb = {
    "model__learning_rate": [0.03, 0.1],
    "model__max_iter": [100],
    "model__max_leaf_nodes": [31, 127],
    "model__l2_regularization": [0.0, 1.0],
}

param_dist_mam = {
    "model__d_model": randint(32, 128),
    "model__n_layers": randint(2, 10),
    "model__lr": uniform(1e-5, 1e-3),
}

param_dist_ftt = {
    "model__d_model": randint(32, 128),
    "model__n_layers": randint(2, 10),
    "model__lr": uniform(1e-5, 1e-3),
}


In [14]:
def _run_search_with_time(name, searcher, fit_params=None):
    """Helper to run sklearn search objects with timing and metrics."""
    fit_params = fit_params or {}

    # Train
    t0 = time.perf_counter()
    searcher.fit(X_train, y_train, **fit_params)
    train_time = time.perf_counter() - t0

    # Predict
    t1 = time.perf_counter()
    y_pred = searcher.predict(X_test)
    pred_time = time.perf_counter() - t1

    # Anzahl Klassen prüfen
    n_classes = len(np.unique(y_test))
    avg = "binary" if n_classes == 2 else "weighted"

    # Metriken
    acc = accuracy_score(y_test, y_pred)
    bacc = balanced_accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average=avg)

    # ROC-AUC
    try:
        y_proba = searcher.predict_proba(X_test)
        if n_classes == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            # One-vs-rest, gewichteter Durchschnitt
            auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")
    except Exception:
        auc = np.nan

    print(f"\n{name}")
    print("-" * 60)
    print(f"Best params:          {searcher.best_params_}")
    print(f"CV best bal-acc:      {searcher.best_score_:.4f}")
    print(f"Test Accuracy:        {acc:.4f}")
    print(f"Test Balanced Acc.:   {bacc:.4f}")
    print(f"Test F1 ({avg}):      {f1:.4f}")
    if not np.isnan(auc):
        print(f"Test ROC-AUC:         {auc:.4f}")
    else:
        print(f"Test ROC-AUC:         n/a")
    print(f"Train time (s):       {train_time:.2f}")
    print(f"Predict time (s):     {pred_time:.4f}")

    return {
        "name": name,
        "grid": searcher,
        "acc": acc,
        "bacc": bacc,
        "f1": f1,
        "auc": auc,
        "train_time": train_time,
        "pred_time": pred_time,
        "cv_best_bal_acc": searcher.best_score_,
    }


def run_grid_clf_with_time(name, pipe, grid, fit_params=None):
    """GridSearchCV für Klassifikation, inkl. Zeitmessung.
    Funktioniert für binary UND multiclass."""
    searcher = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1,
        verbose=0,
    )
    return _run_search_with_time(name, searcher, fit_params=fit_params)


def run_randomized_clf_with_time(
    name, pipe, param_distributions, n_iter=20, random_state=42, fit_params=None
):
    """RandomizedSearchCV Variante für große Suchräume."""
    searcher = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1,
        verbose=0,
        random_state=random_state,
    )
    return _run_search_with_time(name, searcher, fit_params=fit_params)



In [15]:
results = []

#results.append(run_grid_clf_with_time("LR",  pipe_lr,  param_grid_lr))
#results.append(run_grid_clf_with_time("RF",  pipe_rf,  param_grid_rf))
#results.append(run_grid_clf_with_time("HGB", pipe_hgb, param_grid_hgb))

Die Modelle haben leider ein Bug bei der Nutzung von GridSearch, daher wird kein Hyperparameter Tuning angewendet

In [16]:
results.append(run_randomized_clf_with_time("MAM", pipe_mam, param_dist_mam))
results.append(run_randomized_clf_with_time("FTT", pipe_ftt, param_dist_ftt))

Numerical Feature: Annual_Income, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Monthly_Inhand_Salary, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Changed_Credit_Limit, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Outstanding_Debt, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_Utilization_Ratio, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_History_Age, Info: {

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU avail

Numerical Feature: Annual_Income, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Monthly_Inhand_Salary, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Changed_Credit_Limit, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Outstanding_Debt, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_Utilization_Ratio, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_History_Age, Info: {

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU avail

Numerical Feature: Annual_Income, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Monthly_Inhand_Salary, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Changed_Credit_Limit, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Outstanding_Debt, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_Utilization_Ratio, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_History_Age, Info: {

...

Numerical Feature: Annual_Income, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Monthly_Inhand_Salary, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Changed_Credit_Limit, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Outstanding_Debt, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_Utilization_Ratio, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_History_Age, Info: {

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


Numerical Feature: Annual_Income, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Monthly_Inhand_Salary, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Changed_Credit_Limit, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Outstanding_Debt, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_Utilization_Ratio, Info: {'preprocessing': 'imputer -> minmax -> standardization', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: Credit_History_Age, Info: {

Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


ValueError: 
All the 100 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Users/antonioaleksic/Documents/MALE01/.venv/lib/python3.13/site-packages/sklearn/pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^

In [ ]:
df_results = pd.DataFrame(results)

family_map = {
    "LR":  "klassisch",
    "RF":  "klassisch",
    "HGB": "klassisch",
    "MAM": "generativ",
    "FTT": "generativ",
}

df_results["family"] = df_results["name"].map(family_map)

print("\nEinzelne Modelle:")
display(df_results[["name", "family", "bacc", "f1", "auc", "train_time", "pred_time"]])

print("\nGruppensicht nach Modellfamilie:")
df_family = (
    df_results
    .groupby("family")[["bacc", "f1", "auc", "train_time", "pred_time"]]
    .mean()
    .sort_values("bacc", ascending=False)
)
display(df_family)